# COMP 3610: Big Data Analytics - Assignment 4
# MLOps & Model Deployment

---

**Student ID:** 816034871 Kaveesh Ramsarran  
**Course:** COMP 3610 - Big Data Analytics  
**Semester:** II, 2025-2026  

---

## Overview

This notebook documents the full MLOps pipeline for deploying a taxi tip prediction model:

1. **Part 1:** MLflow Experiment Tracking - Logging, comparing, and registering models
2. **Part 2:** FastAPI Model Serving - Building and testing a REST API
3. **Part 3:** Docker Containerization - Packaging the service for deployment
4. **Part 4:** Documentation & Code Quality

We reuse the NYC Yellow Taxi dataset and models from Assignment 2 to predict `tip_amount`.

## Setup and Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import os
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")

All libraries imported successfully!


## Data Loading & Preprocessing

We download the NYC Yellow Taxi Trip Records (January 2024) and apply the same preprocessing and feature engineering from Assignment 2.

In [2]:
# Download taxi trip data
DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

print("Loading NYC Yellow Taxi Trip data...")
df = pd.read_parquet(DATA_URL)
print(f"Loaded {len(df):,} records")

zone_lookup = pd.read_csv(ZONE_LOOKUP_URL)
print(f"Loaded zone lookup with {len(zone_lookup)} zones")

Loading NYC Yellow Taxi Trip data...
Loaded 2,964,624 records
Loaded zone lookup with 265 zones


In [3]:
def clean_taxi_data(df):
    """Clean the taxi data using the same rules as Assignment 2."""
    df_clean = df.copy()
    print(f"Starting records: {len(df_clean):,}")
    
    # Filter credit card payments only
    df_clean = df_clean[df_clean['payment_type'] == 1]
    print(f"After credit card filter: {len(df_clean):,}")
    
    # Valid trip distance
    df_clean = df_clean[(df_clean['trip_distance'] > 0) & (df_clean['trip_distance'] <= 100)]
    
    # Valid fare amount
    df_clean = df_clean[df_clean['fare_amount'] > 0]
    
    # Valid passenger count
    df_clean = df_clean[(df_clean['passenger_count'] >= 1) & (df_clean['passenger_count'] <= 6)]
    
    # Compute trip duration
    df_clean['tpep_pickup_datetime'] = pd.to_datetime(df_clean['tpep_pickup_datetime'])
    df_clean['tpep_dropoff_datetime'] = pd.to_datetime(df_clean['tpep_dropoff_datetime'])
    df_clean['trip_duration_seconds'] = (df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']).dt.total_seconds()
    
    # Valid trip duration (1 min to 3 hours)
    df_clean = df_clean[(df_clean['trip_duration_seconds'] >= 60) & (df_clean['trip_duration_seconds'] <= 10800)]
    
    # Non-negative tips, capped at 100
    df_clean = df_clean[(df_clean['tip_amount'] >= 0) & (df_clean['tip_amount'] <= 100)]
    
    df_clean = df_clean.reset_index(drop=True)
    print(f"Final cleaned dataset: {len(df_clean):,} records")
    return df_clean

df_clean = clean_taxi_data(df)

# Sample for efficient processing (representative subset)
SAMPLE_SIZE = 200_000
df_clean = df_clean.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Sampled {SAMPLE_SIZE:,} records for model training")

Starting records: 2,964,624
After credit card filter: 2,319,046
Final cleaned dataset: 2,268,597 records
Sampled 200,000 records for model training


In [4]:
def engineer_features(df, zone_lookup):
    """Engineer features matching Assignment 2."""
    df_feat = df.copy()
    
    # Temporal features
    df_feat['pickup_hour'] = df_feat['tpep_pickup_datetime'].dt.hour
    df_feat['pickup_day_of_week'] = df_feat['tpep_pickup_datetime'].dt.dayofweek
    df_feat['is_weekend'] = df_feat['pickup_day_of_week'].isin([5, 6]).astype(int)
    
    # Trip features
    df_feat['trip_duration_minutes'] = df_feat['trip_duration_seconds'] / 60.0
    df_feat['trip_speed_mph'] = np.where(
        df_feat['trip_duration_minutes'] > 0,
        df_feat['trip_distance'] / (df_feat['trip_duration_minutes'] / 60.0),
        0
    )
    df_feat['trip_speed_mph'] = df_feat['trip_speed_mph'].clip(upper=100)
    df_feat['log_trip_distance'] = np.log1p(df_feat['trip_distance'])
    
    # Fare features
    df_feat['fare_per_mile'] = np.where(
        df_feat['trip_distance'] > 0,
        df_feat['fare_amount'] / df_feat['trip_distance'],
        0
    )
    df_feat['fare_per_mile'] = df_feat['fare_per_mile'].clip(upper=100)
    df_feat['fare_per_minute'] = np.where(
        df_feat['trip_duration_minutes'] > 0,
        df_feat['fare_amount'] / df_feat['trip_duration_minutes'],
        0
    )
    df_feat['fare_per_minute'] = df_feat['fare_per_minute'].clip(upper=50)
    
    # Zone features - borough encoding
    zone_dict = zone_lookup.set_index('LocationID')['Borough'].to_dict()
    df_feat['pickup_borough'] = df_feat['PULocationID'].map(zone_dict).fillna('Unknown')
    df_feat['dropoff_borough'] = df_feat['DOLocationID'].map(zone_dict).fillna('Unknown')
    
    le_pu = LabelEncoder()
    le_do = LabelEncoder()
    df_feat['pickup_borough_encoded'] = le_pu.fit_transform(df_feat['pickup_borough'])
    df_feat['dropoff_borough_encoded'] = le_do.fit_transform(df_feat['dropoff_borough'])
    
    print(f"Engineered features for {len(df_feat):,} records")
    return df_feat

df_features = engineer_features(df_clean, zone_lookup)

Engineered features for 200,000 records


In [5]:
# Define features and target
FEATURE_COLUMNS = [
    'pickup_hour', 'pickup_day_of_week', 'is_weekend',
    'trip_distance', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance',
    'fare_amount', 'fare_per_mile', 'fare_per_minute',
    'passenger_count',
    'pickup_borough_encoded', 'dropoff_borough_encoded',
    'tolls_amount', 'extra', 'mta_tax', 'congestion_surcharge', 'Airport_fee',
]

print(f"Using {len(FEATURE_COLUMNS)} features:")
for i, feat in enumerate(FEATURE_COLUMNS, 1):
    print(f"  {i:2d}. {feat}")

Using 18 features:
   1. pickup_hour
   2. pickup_day_of_week
   3. is_weekend
   4. trip_distance
   5. trip_duration_minutes
   6. trip_speed_mph
   7. log_trip_distance
   8. fare_amount
   9. fare_per_mile
  10. fare_per_minute
  11. passenger_count
  12. pickup_borough_encoded
  13. dropoff_borough_encoded
  14. tolls_amount
  15. extra
  16. mta_tax
  17. congestion_surcharge
  18. Airport_fee


In [6]:
# Handle missing values
for col in FEATURE_COLUMNS:
    if df_features[col].isnull().any():
        df_features[col] = df_features[col].fillna(df_features[col].median())

# Prepare splits
X = df_features[FEATURE_COLUMNS].values
y = df_features['tip_amount'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training set:   {X_train_scaled.shape[0]:,} samples")
print(f"Validation set: {X_val_scaled.shape[0]:,} samples")
print(f"Test set:       {X_test_scaled.shape[0]:,} samples")

Training set:   140,000 samples
Validation set: 30,000 samples
Test set:       30,000 samples


---

# Part 1: Experiment Tracking with MLflow (25 marks)

We use MLflow to track, compare, and register our regression models from Assignment 2.

## Task 1.1: MLflow Setup & Experiment Logging (10 marks)

We set up a local MLflow tracking server and log at least 2 models with their parameters, metrics, artifacts, and tags.

In [7]:
# Configure MLflow tracking (local file store)
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("taxi-tip-prediction")

print("MLflow experiment 'taxi-tip-prediction' created/set.")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow experiment 'taxi-tip-prediction' created/set.
Tracking URI: mlruns


### Helper function to log regression metrics

In [8]:
def log_regression_metrics(y_true, y_pred):
    """Compute and log MAE, RMSE, and R2 to MLflow."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mlflow.log_metric("mae", round(mae, 4))
    mlflow.log_metric("rmse", round(rmse, 4))
    mlflow.log_metric("r2", round(r2, 4))
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

### Model 1: Linear Regression

We train a Linear Regression baseline and log all relevant information to MLflow.

In [9]:
with mlflow.start_run(run_name="linear-regression-baseline"):
    # Log parameters
    mlflow.log_params({"model_type": "LinearRegression", "n_features": len(FEATURE_COLUMNS)})
    
    # Train model
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    
    # Evaluate on test set
    lr_preds = lr_model.predict(X_test_scaled)
    lr_metrics = log_regression_metrics(y_test, lr_preds)
    
    # Log tags
    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "nyc-taxi-2024-01")
    mlflow.set_tag("author", "Kaveesh Ramsarran")
    
    # Log model artifact
    mlflow.sklearn.log_model(lr_model, "model")
    
    print(f"Linear Regression logged.")
    print(f"  MAE:  ${lr_metrics['MAE']:.4f}")
    print(f"  RMSE: ${lr_metrics['RMSE']:.4f}")
    print(f"  R²:   {lr_metrics['R2']:.4f}")

2026/04/13 12:37:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 12:37:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression logged.
  MAE:  $1.2569
  RMSE: $2.3854
  R²:   0.6066


### Model 2: Random Forest Regressor

We train a Random Forest Regressor (the best-performing model from Assignment 2) and log it to MLflow.

In [10]:
with mlflow.start_run(run_name="random-forest-regressor"):
    # Hyperparameters
    rf_params = {
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_split": 10,
        "min_samples_leaf": 5,
        "random_state": 42,
    }
    mlflow.log_params(rf_params)
    
    # Train model
    rf_model = RandomForestRegressor(**rf_params, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    
    # Evaluate on test set
    rf_preds = rf_model.predict(X_test_scaled)
    rf_metrics = log_regression_metrics(y_test, rf_preds)
    
    # Log tags
    mlflow.set_tag("model_type", "RandomForestRegressor")
    mlflow.set_tag("dataset_version", "nyc-taxi-2024-01")
    mlflow.set_tag("author", "Kaveesh Ramsarran")
    
    # Log model artifact
    mlflow.sklearn.log_model(
        rf_model, "model",
        registered_model_name="taxi-tip-regressor"
    )
    
    rf_run_id = mlflow.active_run().info.run_id
    
    print(f"Random Forest Regressor logged.")
    print(f"  MAE:  ${rf_metrics['MAE']:.4f}")
    print(f"  RMSE: ${rf_metrics['RMSE']:.4f}")
    print(f"  R²:   {rf_metrics['R2']:.4f}")
    print(f"  Run ID: {rf_run_id}")

2026/04/13 12:37:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 12:37:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Regressor logged.
  MAE:  $1.2101
  RMSE: $2.3196
  R²:   0.6280
  Run ID: 1e747d82169e4e93aae00e4158732663


Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
Created version '2' of model 'taxi-tip-regressor'.


**Screenshot: MLflow UI showing all logged runs**

![MLflow Runs](screenshots/mlflow_runs.png)

---

## Task 1.2: Model Comparison & Registry (15 marks)

We compare all logged runs and register the best model.

In [11]:
# Programmatic comparison of all runs
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print("All Logged Runs - Side-by-Side Comparison:")
print("=" * 70)
comparison_cols = ["run_id", "tags.model_type", "metrics.mae", "metrics.rmse", "metrics.r2"]
available_cols = [c for c in comparison_cols if c in runs.columns]
display(runs[available_cols].sort_values("metrics.rmse"))

All Logged Runs - Side-by-Side Comparison:


,run_id,tags.model_type,metrics.mae,metrics.rmse,metrics.r2
0,1e747d82169e4e93aae00e4158732663,RandomForestRegressor,1.2101,2.3196,0.6280
2,8150f8d9bded4eefb7593700e93d9f36,RandomForestRegressor,1.2101,2.3196,0.6280
1,b7510181161e4f578c8ea7abdc7c023f,LinearRegression,1.2569,2.3854,0.6066
3,79f4581936284751b8b00776a121b612,LinearRegression,1.2569,2.3854,0.6066


### Model Comparison Analysis

The **Random Forest Regressor** outperforms Linear Regression across all metrics. It achieves a lower MAE ($1.18 vs $1.25), lower RMSE ($2.27 vs $2.35), and higher R² (0.64 vs 0.61). This is because the Random Forest can capture non-linear relationships between features like fare amount, trip distance, and tip amount, while Linear Regression is limited to linear patterns.

**Screenshot: MLflow comparison view showing side-by-side parameters and metrics**

![MLflow Comparison](screenshots/mlflow_comparison.png)

In [12]:
# Register the best model with a description
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Update version description
try:
    client.update_model_version(
        name="taxi-tip-regressor",
        version=1,
        description=(
            "Random Forest Regressor with 100 trees, max_depth=15. "
            f"Test metrics: MAE=${rf_metrics['MAE']:.4f}, RMSE=${rf_metrics['RMSE']:.4f}, "
            f"R²={rf_metrics['R2']:.4f}. Trained on NYC Yellow Taxi Jan 2024 data."
        )
    )
    print("Model registered as 'taxi-tip-regressor' version 1 with description.")
except Exception as e:
    print(f"Note: {e}")
    print("Model was already registered during the logging step above.")

Model registered as 'taxi-tip-regressor' version 1 with description.


In [13]:
# Load model from registry and make a sample prediction
try:
    registered_model = mlflow.sklearn.load_model("models:/taxi-tip-regressor/1")
    print("Model loaded from MLflow Registry successfully!")
except Exception:
    print("Using locally trained model (MLflow server may not be running).")
    registered_model = rf_model

# Sample prediction
sample = X_test_scaled[:3]
sample_predictions = registered_model.predict(sample)
print("\nSample Predictions:")
for i, (pred, actual) in enumerate(zip(sample_predictions, y_test[:3])):
    print(f"  Trip {i+1}: Predicted=${pred:.2f}, Actual=${actual:.2f}")

Model loaded from MLflow Registry successfully!

Sample Predictions:
  Trip 1: Predicted=$2.90, Actual=$2.94
  Trip 2: Predicted=$2.11, Actual=$2.00
  Trip 3: Predicted=$2.85, Actual=$1.00


### Save Model and Scaler for FastAPI

We save the best model and the fitted scaler as `.joblib` files so the FastAPI application can load them at startup.

In [14]:
# Save model and scaler for the API
os.makedirs("models", exist_ok=True)
joblib.dump(rf_model, "models/model.joblib")
joblib.dump(scaler, "models/scaler.joblib")

print("Saved models/model.joblib")
print("Saved models/scaler.joblib")
print(f"Model file size: {os.path.getsize('models/model.joblib') / 1024 / 1024:.1f} MB")
print(f"Scaler file size: {os.path.getsize('models/scaler.joblib') / 1024:.1f} KB")

Saved models/model.joblib
Saved models/scaler.joblib
Model file size: 36.3 MB
Scaler file size: 1.0 KB


---

# Part 2: Model Serving with FastAPI (35 marks)

We build a REST API that serves predictions from the trained model. The implementation is in `app.py` and tests are in `test_app.py`.

## Task 2.1: API Design & Implementation (15 marks)

The FastAPI application (`app.py`) includes:

- **Model Loading:** Uses the `lifespan` handler to load the model and scaler once at startup
- **POST /predict:** Single prediction endpoint with Pydantic validation
- **Pydantic Input Model:** `TripInput` validates all 18 features with appropriate constraints
- **Response:** Returns `tip_amount`, `model_version`, and `prediction_id` (UUID)

Let's inspect the app code:

In [15]:
# Display the contents of app.py
with open("app.py", "r") as f:
    print(f.read())

"""
COMP 3610 - Assignment 4: FastAPI Prediction Service
Serves tip_amount predictions for NYC Yellow Taxi trips using a trained Random Forest model.
"""

import os
import uuid
import time
import joblib
import numpy as np
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List

# --- Configuration ---
MODEL_PATH = os.getenv("MODEL_PATH", "models/model.joblib")
SCALER_PATH = os.getenv("SCALER_PATH", "models/scaler.joblib")
MODEL_VERSION = "1.0.0"
MODEL_NAME = "taxi-tip-regressor"

# Feature order must match training
FEATURE_COLUMNS = [
    "pickup_hour", "pickup_day_of_week", "is_weekend",
    "trip_distance", "trip_duration_minutes", "trip_speed_mph", "log_trip_distance",
    "fare_amount", "fare_per_mile", "fare_per_minute",
    "passenger_count",
    "pickup_borough_encoded", "dropoff_borough_encoded",
    "tolls_amount", "extra", "mta_tax", "congestion_sur

## Task 2.2: Enhanced API Features (10 marks)

The API includes the following additional endpoints:

- **POST /predict/batch** - Accepts up to 100 trip records, enforced by `max_length=100` on the Pydantic `BatchInput`
- **GET /health** - Returns API status, model loaded status, model version, and uptime
- **GET /model/info** - Returns model name, version, feature list, and training metrics
- **Global exception handler** - Catches all unhandled exceptions and returns structured HTTP 500 without exposing internals

## Task 2.3: API Testing (10 marks)

We have 12 test cases in `test_app.py` covering:
1. Successful single prediction with valid input
2. Successful batch prediction
3. Invalid input rejection (missing fields, wrong types, out-of-range values)
4. Health check endpoint
5. Edge cases (zero distance, extreme fare values, batch limit)

In [16]:
# Display the test file
with open("test_app.py", "r") as f:
    print(f.read())

"""
COMP 3610 - Assignment 4: API Test Suite
Tests for the Taxi Tip Prediction FastAPI application.
"""

import pytest
from fastapi.testclient import TestClient
from app import app


@pytest.fixture(scope="module")
def client():
    with TestClient(app) as c:
        yield c

# --- Valid sample input ---
VALID_INPUT = {
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "is_weekend": 0,
    "trip_distance": 3.5,
    "trip_duration_minutes": 15.0,
    "trip_speed_mph": 14.0,
    "log_trip_distance": 1.504,
    "fare_amount": 18.0,
    "fare_per_mile": 5.14,
    "fare_per_minute": 1.2,
    "passenger_count": 1,
    "pickup_borough_encoded": 3,
    "dropoff_borough_encoded": 3,
    "tolls_amount": 0.0,
    "extra": 1.0,
    "mta_tax": 0.5,
    "congestion_surcharge": 2.5,
    "Airport_fee": 0.0,
}


# --- Happy path tests ---

def test_root(client):
    """Root endpoint returns 200."""
    response = client.get("/")
    assert response.status_code == 200
    assert "message" in respo

In [17]:
# Run the tests
import subprocess
result = subprocess.run(["python", "-m", "pytest", "test_app.py", "-v"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.14.0, pytest-9.0.2, pluggy-1.6.0 -- C:\Program Files\Python314\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\User\816034871_COMP3610_A4
plugins: anyio-4.12.1
collecting ... collected 12 items

test_app.py::test_root PASSED                                            [  8%]
test_app.py::test_health PASSED                                          [ 16%]
test_app.py::test_predict_valid PASSED                                   [ 25%]
test_app.py::test_batch_prediction PASSED                                [ 33%]
test_app.py::test_model_info PASSED                                      [ 41%]
test_app.py::test_predict_missing_field PASSED                           [ 50%]
test_app.py::test_predict_invalid_type PASSED                            [ 58%]
test_app.py::test_predict_out_of_range_pickup_hour PASSED                [ 66%]
test_app.py::test_predict_negative_distance PASSED   

**Screenshot: Swagger UI auto-generated API documentation at /docs**

![Swagger UI](screenshots/swagger_ui.png)

To start the API locally:
```bash
uvicorn app:app --reload --port 8000
```

---

# Part 3: Containerization with Docker (20 marks)

We containerize the prediction service using Docker and orchestrate it with Docker Compose.

## Task 3.1: Dockerfile & Image Building (10 marks)

Our Dockerfile uses `python:3.11-slim` as the base image, copies only necessary files, installs dependencies, and starts uvicorn.

In [18]:
# Display Dockerfile
with open("Dockerfile", "r") as f:
    print(f.read())

# 1. Start from a slim Python base image to keep the container small (~150MB vs ~900MB for full)
FROM python:3.11-slim

# 2. Set the working directory inside the container
WORKDIR /app

# 3. Copy dependency file first for Docker layer caching
#    If requirements.txt hasn't changed, Docker reuses the cached pip install layer
COPY requirements.txt .

# 4. Install Python dependencies without caching to reduce image size
RUN pip install --no-cache-dir -r requirements.txt

# 5. Copy application code and model artifacts
COPY app.py .
COPY models/ ./models/

# 6. Document which port the app exposes (does not actually publish it)
EXPOSE 8000

# 7. Start the FastAPI server using uvicorn
#    --host 0.0.0.0 is required so the container is reachable from outside
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



In [19]:
# Display .dockerignore
with open(".dockerignore", "r") as f:
    print(f.read())

# Byte-compiled / optimized / DLL files
__pycache__/
*.pyc
*.pyo

# Version control
.git/
.gitignore

# Data files (not needed in container)
data/
*.parquet
*.csv

# MLflow tracking data
mlruns/

# Environment files
.env
.venv/
venv/

# Jupyter notebooks and checkpoints
*.ipynb
.ipynb_checkpoints/

# Documentation
*.md

# Test files (not needed in production container)
test_*.py

# IDE settings
.vscode/
.idea/

# OS files
.DS_Store
Thumbs.db

# pytest cache
.pytest_cache/



In [21]:
# Build the Docker image
import subprocess
import shutil

docker_available = shutil.which("docker") is not None

if docker_available:
    result = subprocess.run(
        ["docker", "build", "-t", "taxi-tip-api", "."],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print(result.stdout)
    if result.returncode != 0:
        print("Docker build failed:", result.stderr)
    else:
        print("Docker image built successfully!")
else:
    print("Docker is not installed on this system.")
    print("To build the image, install Docker Desktop and run:")
    print("  docker build -t taxi-tip-api .")


Docker image built successfully!


In [22]:
# Report image size
if docker_available:
    result = subprocess.run(["docker", "images", "taxi-tip-api"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout)
else:
    print("Docker not available - skipping image size report.")

IMAGE                 ID             DISK USAGE   CONTENT SIZE   EXTRA
taxi-tip-api:latest   8044dd8370d0       1.31GB          299MB        



In [23]:
# Run the container and test it
if docker_available:
    result = subprocess.run(
        ["docker", "run", "-d", "--name", "taxi-api-test", "-p", "8000:8000", "taxi-tip-api"],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print(result.stdout)
    if result.returncode != 0:
        print("Docker run failed:", result.stderr)
    import time
    time.sleep(5)
else:
    print("Docker not available - skipping container test.")

64df8097874113d1d03e8d0fe9ebda78ae739bac69dab5565541d4dca0e3367f



In [24]:
# Test the containerized API
import requests
import json

if docker_available:
    try:
        # Health check
        response = requests.get("http://localhost:8000/health")
        print("Health Check:")
        print(json.dumps(response.json(), indent=2))

        # Make a prediction
        test_payload = {
            "pickup_hour": 14, "pickup_day_of_week": 2, "is_weekend": 0,
            "trip_distance": 3.5, "trip_duration_minutes": 15.0, "trip_speed_mph": 14.0,
            "log_trip_distance": 1.504, "fare_amount": 18.0, "fare_per_mile": 5.14,
            "fare_per_minute": 1.2, "passenger_count": 1, "pickup_borough_encoded": 3,
            "dropoff_borough_encoded": 3, "tolls_amount": 0.0, "extra": 1.0,
            "mta_tax": 0.5, "congestion_surcharge": 2.5, "Airport_fee": 0.0,
        }
        response = requests.post("http://localhost:8000/predict", json=test_payload)
        print("\nPrediction:")
        print(json.dumps(response.json(), indent=2))
    except requests.ConnectionError:
        print("Could not connect to containerized API.")
else:
    print("Docker not available - skipping containerized API test.")

Health Check:
{
  "status": "healthy",
  "model_loaded": true,
  "model_version": "1.0.0",
  "uptime_seconds": 10.9
}

Prediction:
{
  "tip_amount": 3.99,
  "prediction_id": "feb949dd-5eb5-471c-be4f-90c7e54ef1ca",
  "model_version": "1.0.0"
}


In [25]:
# Clean up the test container
subprocess.run(["docker", "stop", "taxi-api-test"], capture_output=True, text=True)
subprocess.run(["docker", "rm", "taxi-api-test"], capture_output=True, text=True)
print("Test container cleaned up.")

Test container cleaned up.


## Task 3.2: Docker Compose & Deployment Demo (10 marks)

Our `docker-compose.yml` defines two services:
1. **api** - The FastAPI prediction service built from the Dockerfile
2. **mlflow** - An MLflow tracking server (bonus)

The API service uses environment variables for configuration and depends on the MLflow service.

In [26]:
# Display docker-compose.yml
with open("docker-compose.yml", "r") as f:
    print(f.read())

# Docker Compose configuration for the Taxi Tip Prediction service.
# Includes the FastAPI prediction API and an MLflow tracking server (bonus).
# Start all services: docker compose up --build
# Stop all services:  docker compose down

services:
  # FastAPI prediction API
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/model.joblib
      - SCALER_PATH=/app/models/scaler.joblib
      - MLFLOW_TRACKING_URI=http://mlflow:5000
    depends_on:
      - mlflow
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # MLflow tracking server (bonus)
  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.12.1
    ports:
      - "5000:5000"
    command: mlflow server --host 0.0.0.0 --port 5000
    volumes:
      - mlflow-data:/mlflow

volumes:
  mlflow-data:



In [27]:
# Start all services
if docker_available:
    result = subprocess.run(["docker", "compose", "up", "-d", "--build"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.returncode != 0:
        print("Docker Compose failed:", result.stderr)
else:
    print("Docker not available - skipping Docker Compose.")

#1 [internal] load local bake definitions
#1 reading from stdin 545B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 891B done
#2 DONE 0.0s

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#4 [internal] load metadata for docker.io/library/python:3.11-slim
#4 DONE 0.6s

#5 [internal] load .dockerignore
#5 transferring context: 561B done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 218B done
#6 DONE 0.0s

#7 [1/6] FROM docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf
#7 resolve docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf 0.0s done
#7 DONE 0.1s

#8 [2/6] WORKDIR /app
#8 CACHED

#9 [5/6] COPY app.py .
#9 CACHED

#10 [3/6] COPY requirements.txt .
#10 CACHED

#11 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#11 CACHED

#12 [6/6] COPY models/ ./models/
#12 CA

In [28]:
# Wait for services and check running containers
if docker_available:
    import time
    time.sleep(10)
    result = subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout)
else:
    print("Docker not available - skipping container check.")

NAME                             IMAGE                           COMMAND                  SERVICE   CREATED          STATUS                             PORTS
816034871_comp3610_a4-api-1      816034871_comp3610_a4-api       "uvicorn app:app --h…"   api       18 seconds ago   Up 16 seconds (health: starting)   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
816034871_comp3610_a4-mlflow-1   ghcr.io/mlflow/mlflow:v2.12.1   "mlflow server --hos…"   mlflow    18 seconds ago   Up 17 seconds                      0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp



In [29]:
import requests
import json

BASE_URL = "http://localhost:8000"

if docker_available:
    try:
        # Prediction 1: Short Manhattan trip
        pred1 = requests.post(f"{BASE_URL}/predict", json={
            "pickup_hour": 8, "pickup_day_of_week": 0, "is_weekend": 0,
            "trip_distance": 1.5, "trip_duration_minutes": 10.0, "trip_speed_mph": 9.0,
            "log_trip_distance": 0.916, "fare_amount": 10.0, "fare_per_mile": 6.67,
            "fare_per_minute": 1.0, "passenger_count": 1, "pickup_borough_encoded": 3,
            "dropoff_borough_encoded": 3, "tolls_amount": 0.0, "extra": 1.0,
            "mta_tax": 0.5, "congestion_surcharge": 2.5, "Airport_fee": 0.0
        })
        print("Prediction 1 (Short Manhattan trip):")
        print(json.dumps(pred1.json(), indent=2))

        # Prediction 2: Airport trip
        pred2 = requests.post(f"{BASE_URL}/predict", json={
            "pickup_hour": 18, "pickup_day_of_week": 4, "is_weekend": 0,
            "trip_distance": 17.0, "trip_duration_minutes": 45.0, "trip_speed_mph": 22.7,
            "log_trip_distance": 2.890, "fare_amount": 52.0, "fare_per_mile": 3.06,
            "fare_per_minute": 1.16, "passenger_count": 2, "pickup_borough_encoded": 3,
            "dropoff_borough_encoded": 4, "tolls_amount": 6.55, "extra": 1.0,
            "mta_tax": 0.5, "congestion_surcharge": 2.5, "Airport_fee": 1.75
        })
        print("\nPrediction 2 (Airport trip):")
        print(json.dumps(pred2.json(), indent=2))

        # Prediction 3: Weekend evening trip
        pred3 = requests.post(f"{BASE_URL}/predict", json={
            "pickup_hour": 22, "pickup_day_of_week": 5, "is_weekend": 1,
            "trip_distance": 5.0, "trip_duration_minutes": 20.0, "trip_speed_mph": 15.0,
            "log_trip_distance": 1.792, "fare_amount": 25.0, "fare_per_mile": 5.0,
            "fare_per_minute": 1.25, "passenger_count": 3, "pickup_borough_encoded": 3,
            "dropoff_borough_encoded": 1, "tolls_amount": 0.0, "extra": 2.5,
            "mta_tax": 0.5, "congestion_surcharge": 2.5, "Airport_fee": 0.0
        })
        print("\nPrediction 3 (Weekend evening trip):")
        print(json.dumps(pred3.json(), indent=2))
    except requests.ConnectionError:
        print("Could not connect to Docker Compose services.")
else:
    print("Docker not available - skipping deployment demo.")

Prediction 1 (Short Manhattan trip):
{
  "tip_amount": 2.98,
  "prediction_id": "4ae9c68b-a89d-4d27-a4ff-05fbd7280633",
  "model_version": "1.0.0"
}

Prediction 2 (Airport trip):
{
  "tip_amount": 4.23,
  "prediction_id": "a2203f3a-0448-4234-b9d2-2c97b48b022c",
  "model_version": "1.0.0"
}

Prediction 3 (Weekend evening trip):
{
  "tip_amount": 5.51,
  "prediction_id": "4895e8ad-9aec-4e5a-8945-8a122987a71b",
  "model_version": "1.0.0"
}


In [30]:
# Shut down cleanly
if docker_available:
    result = subprocess.run(["docker", "compose", "down"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout)
print("All services stopped.")


All services stopped.


In [31]:
# Document container size
if docker_available:
    result = subprocess.run(
        ["docker", "images", "taxi-tip-api", "--format", "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print(result.stdout)
else:
    print("Docker not available - image size will be reported after Docker build.")

REPOSITORY     TAG       SIZE
taxi-tip-api   latest    1.31GB



---

# Part 4: Documentation & Code Quality (10 marks)

## Project Organization

- **README.md** - Setup instructions, prerequisites, and step-by-step guide
- **requirements.txt** - All Python dependencies with minimum versions
- **.gitignore** - Excludes data files, mlruns/, model artifacts, and __pycache__/
- **.dockerignore** - Excludes unnecessary files from the Docker build context

## Code Quality

- Meaningful variable and function names throughout all files
- Reusable `_predict_single()` helper used by both `/predict` and `/predict/batch`
- `log_regression_metrics()` helper to avoid repeated metric logging code
- Comments explaining Dockerfile layer caching strategy and Docker Compose networking

In [33]:
# Display README
with open("README.md", "r", encoding="utf-8") as f:
    print(f.read())

# COMP 3610 - Assignment 4: MLOps & Model Deployment

**Student:** Kaveesh Ramsarran (816034871)  
**Course:** COMP 3610 - Big Data Analytics  
**Semester:** II, 2025-2026

## Overview

This project deploys a taxi tip prediction model as a containerized REST API. It covers:

1. **MLflow Experiment Tracking** - Logging, comparing, and registering models
2. **FastAPI Prediction Service** - REST API with input validation and error handling
3. **Docker Containerization** - Dockerfile and Docker Compose for reproducible deployment

The model predicts `tip_amount` for NYC Yellow Taxi trips using a Random Forest Regressor trained on January 2024 data (from Assignment 2).

## Prerequisites

- Python 3.10+
- Docker Desktop installed and running
- pip package manager

## Quick Start

### 1. Install Dependencies

```bash
pip install -r requirements.txt
```

### 2. Run the Notebook

Open `assignment4.ipynb` and run all cells. This will:
- Download NYC Taxi data and train models
- Log experiments t

In [34]:
# Display requirements.txt
with open("requirements.txt", "r", encoding="utf-8") as f:
    print(f.read())

mlflow>=2.12.0
fastapi>=0.110.0
uvicorn>=0.29.0
pydantic>=2.0.0
httpx>=0.27.0
scikit-learn>=1.4.0
joblib>=1.3.0
pandas>=2.2.0
numpy>=1.26.0
pytest>=8.0.0
pyarrow>=15.0.0



---

## AI Tools Used

In accordance with the course policy on AI disclosure, the following AI tools were used during the completion of this assignment:

1. **GitHub Copilot (Claude):** Used for:
   - Dockerfile and docker-compose.yml configuration
   - Debugging and troubleshooting Docker networking issues

2. **Purpose of AI usage:** All generated code was reviewed, understood, and adapted to fit the specific requirements of this assignment. The AI tools helped accelerate development while maintaining understanding of the underlying concepts.